# Kytos k004 — Kaggle smoke: real control resampling + ContextConditioned Layer A

> **Ratiocine two-phase pattern** for Kaggle free tier (13GB RAM CPU, 16GB T4×2, 9h/session). Local 8GB Mac can't run full `vcc prep` / `cell-eval --ceiling` — this notebook proves the next baseline before we burn paid compute.

**What this proves (in ~5 min for 20 targets × 3 contexts):**
- `src/kytos/features/basal.py` + `src/kytos/models/layer_a.py` + `layer_b.py` work on real 2026 controls (sparse-safe, no densify of 26GB dense)
- **Exp A (k004-resample):** real control-cell resampling baseline — samples *actual* control cells with replacement per perturbation (preserves biological dispersion, unlike k003 sparse top-300 mean)
- **Exp B (k004-layer_a_b):** `ContextConditionedTransfer` (basal rank-conditioned) + `AdditiveTransportSampler` — first non-trivial gene-transfer model
- Writes `pred_resample.h5ad` / `pred_layer_a_b.h5ad` cell-eval-ready (`target_gene`/`context`, sparse, gzip)

**Inputs:** Kaggle dataset `vcc2026-controls` (632MB: `context_A/B/C.h5ad` + `gene_names.csv`/`pert_counts.csv`) or local `data/raw/vcc2026/`. See `tools/kaggle_bundle.py` for dataset creation.

**Next after this smoke:** if `layer_a_b` shifts target genes in the right direction (sanity check below) and dispersion looks biological, promote to full 300-target run on Vast/RunPod 32GB + `vcc prep --dry-run` → `vcc submit` (daily 2-slot guardrail).


## 0. Setup — install + GPU check

Kaggle images already have `anndata`/`scanpy`/`scipy`; we add `cell-eval` only for local eval. Skip the `pip install` cell if you're re-running.


In [ ]:
# Kaggle setup: offline-first install (wheels vendored for space-constrained Mac + flaky DNS)
import pathlib, subprocess, sys
wheels = pathlib.Path("wheels")
if not wheels.exists():
    # when running from /kaggle/working/kytos or /kaggle/input, try sibling
    import pathlib as _pl
    for cand in [pathlib.Path("/kaggle/working/wheels"), pathlib.Path("wheels"), pathlib.Path("/tmp/kaggle-k004-kernel/wheels")]:
        if cand.exists():
            wheels = cand; break
print(f"wheels dir exists={wheels.exists()} {wheels}")
if wheels.exists() and any(wheels.iterdir()):
    print("installing from vendored wheels --no-index ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", f"--find-links={wheels}", "-q", "anndata", "scanpy", "h5py"])
else:
    print("wheels not found, trying online pip with retries...")
    for attempt in range(3):
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anndata", "scanpy", "h5py", "--retries", "5", "--timeout", "100"])
            break
        except subprocess.CalledProcessError as e:
            print(f"attempt {attempt+1} failed: {e}")
            if attempt==2: raise
import sys
print(sys.version)
try:
    import torch
    print(f"torch {torch.__version__} cuda={torch.cuda.is_available()} device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
except Exception as e:
    print(f"torch not available: {e}")
import numpy, scipy, anndata, pandas
print(f"numpy {numpy.__version__} scipy {scipy.__version__} anndata {anndata.__version__}")


In [ ]:
# Clone repo if running as pure Kaggle notebook (no repo attached)
# If you attached the Kytos GitHub repo as a Kaggle dataset, skip this.
import pathlib, os
REPO = pathlib.Path("/kaggle/working/kytos") if pathlib.Path("/kaggle").exists() else pathlib.Path.cwd()
if not (REPO / "src" / "kytos" / "features" / "basal.py").exists():
    if pathlib.Path("/kaggle").exists():
        print("cloning kytos...")
        import subprocess
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/udirobert/kytos.git", str(REPO)])
        print(f"cloned to {REPO}")
    else:
        REPO = pathlib.Path.cwd().parent if (pathlib.Path.cwd().parent / "src" / "kytos").exists() else pathlib.Path.cwd()
        print(f"local REPO={REPO}")
else:
    print(f"REPO={REPO} exists")
import sys
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
print("sys.path[0]:", sys.path[0])


## 1. Locate VCC 2026 controls


In [ ]:
import pathlib
from pathlib import Path
import pandas as pd

def resolve_raw():
    for p in [
        Path("/kaggle/input/vcc2026-controls"),
        Path("/kaggle/input/kytos-vcc2026-controls"),
        Path("data/raw/vcc2026"),
        REPO / "data" / "raw" / "vcc2026",
        Path("/tmp/kaggle-vcc2026-controls"),
    ]:
        if (p / "gene_names.csv").exists():
            return p
        if p.exists():
            for sub in p.rglob("gene_names.csv"):
                return sub.parent
    raise FileNotFoundError("gene_names.csv not found. Upload vcc2026-controls dataset or set RAW_DIR.")

RAW = resolve_raw()
print(f"RAW={RAW}")
gene_order = pd.read_csv(RAW / "gene_names.csv", header=None, skiprows=1)[0].tolist()
all_targets = pd.read_csv(RAW / "pert_counts.csv", header=None, skiprows=1)[0].tolist()
import json
manifest = json.loads((RAW / "manifest.json").read_text())
print(f"genes={len(gene_order)} targets={len(all_targets)} manifest={manifest['panel_id']} per_context_cells={manifest['per_context']['A']['control_cells']}")
print(f"first 5 genes: {gene_order[:5]}")
print(f"first 5 targets: {all_targets[:5]}")


## 2. EDA — per-context basal stats (sparse-safe)

Validates `src/kytos/features/basal.py` without densifying 26GB dense.


In [ ]:
import anndata as ad
import numpy as np
from kytos.features.basal import extract_basal_context

for ctx in ["A","B","C"]:
    path = RAW / f"context_{ctx}.h5ad"
    if not path.exists():
        path = next(RAW.rglob(f"context_{ctx}.h5ad"))
    adata = ad.read_h5ad(str(path))
    print(f"\n[{ctx}] {adata.n_obs} x {adata.n_vars}  nnz={adata.X.nnz:,}  nnz/cell={adata.X.nnz/adata.n_obs:.1f}  sparsity={1-adata.X.nnz/(adata.n_obs*adata.n_vars):.4f}  X dtype={adata.X.dtype}")
    print(f"  obs: {adata.obs.columns.tolist()}  var: {adata.var.columns.tolist()[:4]}")
    ctx_stats = extract_basal_context(adata.X, gene_order)
    top = np.argsort(ctx_stats.mean_expression)[-5:][::-1]
    print(f"  top5 by control mean: {[(gene_order[i], round(float(ctx_stats.mean_expression[i]),1), round(float(ctx_stats.sparsity[i]),2)) for i in top]}")
    print(f"  mean expression: min {ctx_stats.mean_expression.min():.3f}  median {np.median(ctx_stats.mean_expression):.3f}  max {ctx_stats.mean_expression.max():.1f}")


## 3. Build predictions — Exp A (resample) + Exp B (Layer A+B)

Memory: each 400-cell slice densified alone = 400 × 18533 × 4B ≈ 29MB. Full 360k × 18533 stays sparse (~0.5–8GB depending on mode).


In [ ]:
import anndata as ad
import numpy as np, pandas as pd
from scipy import sparse
from kytos.features.basal import extract_basal_context
from kytos.models.layer_a import ContextConditionedTransfer
from kytos.models.layer_b import AdditiveTransportSampler
import time

PERT_COL, CONTEXT_COL = "target_gene", "context"
MAX_TARGETS = 20  # Kaggle free: 20-30 fits 13GB; set 300 for full 360k run on 32GB Vast
CELLS_PER_PERT = 400
CONTEXTS = ["A","B","C"]
SEED = 0
targets = all_targets[:MAX_TARGETS]
print(f"targets={len(targets)} contexts={CONTEXTS} cells/pert={CELLS_PER_PERT} -> total ~{len(targets)*len(CONTEXTS)*CELLS_PER_PERT:,} cells")

def build_one_context(context, path, targets, gene_order, cells_per_pert, mode, seed):
    ctrl = ad.read_h5ad(str(path))
    ctx = extract_basal_context(ctrl.X, gene_order)
    rng = np.random.default_rng(seed)
    model = ContextConditionedTransfer() if mode=="layer_a_b" else None
    sampler = AdditiveTransportSampler(noise_scale=0.05) if mode=="layer_a_b" else None
    X_csr = ctrl.X.tocsr() if not sparse.isspmatrix_csr(ctrl.X) else ctrl.X
    blocks, obss = [], []
    for tgt in targets:
        if mode=="resample":
            idx = rng.choice(ctx.n_cells, size=cells_per_pert, replace=True)
            Xb = X_csr[idx]
        else:
            delta = model.predict_delta(tgt, ctx)
            idx = rng.choice(ctx.n_cells, size=cells_per_pert, replace=True)
            basal = np.asarray(X_csr[idx].todense(), dtype=np.float32)
            perturbed = sampler.sample_cells(basal, delta.astype(np.float32), n_samples=cells_per_pert, seed=int(rng.integers(0,1_000_000)))
            Xb = sparse.csr_matrix(perturbed)
        blocks.append(Xb)
        obss.append(pd.DataFrame({PERT_COL:[tgt]*cells_per_pert, CONTEXT_COL:[context]*cells_per_pert}))
    return sparse.vstack(blocks, format="csr"), pd.concat(obss, ignore_index=True)

def build_all(mode):
    t0=time.time()
    blocks, obss=[],[]
    for ctx in CONTEXTS:
        p = RAW / f"context_{ctx}.h5ad"
        if not p.exists():
            p = next(RAW.rglob(f"context_{ctx}.h5ad"))
        print(f"[{ctx}] mode={mode} ...", flush=True)
        Xb, obs = build_one_context(ctx, p, targets, gene_order, CELLS_PER_PERT, mode, seed=SEED)
        print(f"  {ctx}: {Xb.shape[0]} cells nnz {Xb.nnz:,}")
        blocks.append(Xb); obss.append(obs)
    X_all = sparse.vstack(blocks, format="csr")
    obs_all = pd.concat(obss, ignore_index=True)
    var = pd.DataFrame(index=pd.Index(gene_order, name="gene_name"))
    adata = ad.AnnData(X=X_all, obs=obs_all, var=var)
    adata.obs[PERT_COL]=adata.obs[PERT_COL].astype("category")
    adata.obs[CONTEXT_COL]=adata.obs[CONTEXT_COL].astype("category")
    print(f"[{mode}] done {adata.n_obs} x {adata.n_vars} nnz {adata.X.nnz:,} in {time.time()-t0:.1f}s")
    return adata

adata_resample = build_all("resample")
adata_layer = build_all("layer_a_b")


## 4. Sanity — does Layer A actually knock down target genes?

Checks that `ContextConditionedTransfer` moves the target gene mean vs the resample baseline, and that the shift magnitude correlates with basal rank (barely-expressed targets attenuate).


In [ ]:
import numpy as np
from scipy import sparse

def mean_for_target(adata, tgt):
    m = adata.obs[PERT_COL]==tgt
    X = adata[m].X
    return np.asarray(X.mean(axis=0)).ravel() if sparse.issparse(X) else X.mean(axis=0)

for tgt in targets[:5]:
    if tgt not in gene_order: continue
    gi = gene_order.index(tgt)
    m_r = mean_for_target(adata_resample, tgt)[gi]
    m_l = mean_for_target(adata_layer, tgt)[gi]
    delta = m_l - m_r
    print(f"{tgt:8s}  resample mean {m_r:6.2f}  layer_a_b mean {m_l:6.2f}  delta {delta:+6.2f}  {'✓ knockdown' if delta < -0.5 else '· weak'}")


## 5. Write cell-eval-ready H5ADs


In [ ]:
import pathlib, json, time
OUT = pathlib.Path("/kaggle/working/k004") if pathlib.Path("/kaggle").exists() else pathlib.Path("experiments/k004-kaggle-smoke")
OUT.mkdir(parents=True, exist_ok=True)
for adata, mode in [(adata_resample,"resample"),(adata_layer,"layer_a_b")]:
    p = OUT / f"pred_{mode}.h5ad"
    print(f"writing {p} ...")
    adata.write_h5ad(str(p), compression="gzip")
    print(f"  {p.stat().st_size/1e6:.1f} MB")
    meta={"run_id":f"k004-kaggle-smoke-{mode}","created":time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),"mode":mode,"n_targets":len(targets),"contexts":CONTEXTS,"cells_per_pert":CELLS_PER_PERT,"total_cells":int(adata.n_obs),"n_genes":int(adata.n_vars),"seed":SEED}
    (OUT / f"meta_{mode}.json").write_text(json.dumps(meta, indent=2))
    print(f"  meta -> {OUT / f'meta_{mode}.json'}")
print(f"\nOUT={OUT}")
import os
print(os.listdir(OUT))


## 6. (Optional) cell-eval proxy + `vcc prep --dry-run`

Full `cell-eval run` needs withheld ground truth (not in this dataset). This cell shows how to validate the H5AD against `vcc` if you have a `.vcc` bundle, and runs a local pseudobulk MSE proxy. On a 32GB Vast run, replace with `vcc prep` + `vcc submit`.


In [ ]:
import numpy as np
from scipy import sparse
def pseudobulk(adata):
    return np.asarray(adata.X.mean(axis=0)).ravel() if sparse.issparse(adata.X) else adata.X.mean(axis=0)
pb_r = pseudobulk(adata_resample)
pb_l = pseudobulk(adata_layer)
mse = ((pb_r - pb_l)**2).mean()
print(f"pseudobulk MSE resample vs layer_a_b: {mse:.4f}")
diff = pb_l - pb_r
top = np.argsort(np.abs(diff))[-10:][::-1]
print("top shifted genes (layer_a_b - resample):")
for i in top:
    print(f"  {gene_order[i]:10s} {diff[i]:+7.3f}  (resample {pb_r[i]:.2f} -> layer {pb_l[i]:.2f})")
print("\nIf top shifted genes are the perturbed targets, Layer A is wired correctly.")


In [ ]:
import shutil, subprocess
if shutil.which("vcc"):
    print("vcc found, trying dry-run on pred_layer_a_b.h5ad ...")
    try:
        subprocess.run(["vcc", "prep", "--help"], check=False)
    except Exception as e:
        print(e)
else:
    print("vcc not installed in this kernel (expected on Kaggle). Validate locally via:")
    print("  vcc prep --dry-run experiments/k004-kaggle-smoke/pred_layer_a_b.h5ad")


## 7. Next steps

- **If knockdown sanity is ✓:** bump `MAX_TARGETS=300` + `CONTEXTS=[A,B,C]` → full 360k-cell run on **Vast/RunPod 32GB** (not Kaggle free — 300×400×3 = 360k cells × 18k genes sparse ≈ 8GB data, needs 32GB RAM). Then `vcc prep` → `vcc submit` (≤2/day).
- **If weak:** tune `ContextConditionedTransfer.knockdown_efficiency` / `attenuation_factor` or swap in a learned per-target prior from Replogle multiline.
- **Artifacts to commit (small):** `meta_*.json` + proxy numbers → `experiments/k004-kaggle-smoke/` (H5ADs stay on Kaggle/HF, not git).
- **HF mirror:** `tools/kaggle_bundle.py` already prepares the dataset dir; same files push to `kytos/atlas-2026-processed` via `huggingface_hub`.
